In [1]:
import pandas as pd
import numpy as np
import os

# ==============================
# 1. Load Raw Dataset
# ==============================

df = pd.read_csv(
    "data/market_data_raw.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Original Shape:", df.shape)
print("\nOriginal Missing Values:")
print(df.isnull().sum())


# ==============================
# 2. Handle Missing Values
# ==============================

# Forward fill only:
# Uses previously known market value.
# Avoid bfill because it can introduce future information.

df = df.ffill()

# Remove initial rows if any asset still has no historical value
df = df.dropna()

print("\nAfter Missing Value Handling:")
print(df.isnull().sum())

print("\nShape After Cleaning:", df.shape)


# ==============================
# 3. Daily Returns
# ==============================

for col in ["SP500", "NASDAQ", "GOLD", "OIL"]:
    df[f"{col}_Return"] = df[col].pct_change()


# ==============================
# 4. Log Returns
# ==============================

for col in ["SP500", "NASDAQ", "GOLD", "OIL"]:
    df[f"{col}_Log_Return"] = np.log(
        df[col] / df[col].shift(1)
    )


# ==============================
# 5. Rolling Volatility
# ==============================

# Annualized volatility
df["SP500_Volatility_10"] = (
    df["SP500_Return"].rolling(10).std() * np.sqrt(252)
)

df["SP500_Volatility_20"] = (
    df["SP500_Return"].rolling(20).std() * np.sqrt(252)
)

df["NASDAQ_Volatility_20"] = (
    df["NASDAQ_Return"].rolling(20).std() * np.sqrt(252)
)


# ==============================
# 6. Momentum Features
# ==============================

df["SP500_Momentum_5"] = df["SP500"].pct_change(5)

df["SP500_Momentum_10"] = df["SP500"].pct_change(10)

df["SP500_Momentum_20"] = df["SP500"].pct_change(20)


# ==============================
# 7. Moving Average Distance
# ==============================

df["SP500_MA20"] = df["SP500"].rolling(20).mean()

df["SP500_MA50"] = df["SP500"].rolling(50).mean()

df["SP500_MA20_Distance"] = (
    df["SP500"] / df["SP500_MA20"] - 1
)

df["SP500_MA50_Distance"] = (
    df["SP500"] / df["SP500_MA50"] - 1
)


# ==============================
# 8. VIX Features
# ==============================

df["VIX_Change"] = df["VIX"].pct_change()

df["VIX_MA10"] = df["VIX"].rolling(10).mean()

df["VIX_Ratio"] = df["VIX"] / df["VIX_MA10"]


# ==============================
# 9. Remove Rolling NaNs
# ==============================

df = df.dropna()


# ==============================
# 10. Validation
# ==============================

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nColumns:")
for col in df.columns:
    print(col)

print("\nSample:")
print(df.head())


# ==============================
# 11. Save Dataset
# ==============================

os.makedirs("data", exist_ok=True)

df.to_csv(
    "data/market_data_processed.csv"
)

print(
    "\nSaved: data/market_data_processed.csv"
)

Original Shape: (6680, 5)

Original Missing Values:
SP500       6
VIX         5
NASDAQ      6
GOLD      187
OIL       178
dtype: int64

After Missing Value Handling:
SP500     0
VIX       0
NASDAQ    0
GOLD      0
OIL       0
dtype: int64

Shape After Cleaning: (6513, 5)

Final Dataset Shape:
(6462, 26)

Total Missing Values:
0

Columns:
SP500
VIX
NASDAQ
GOLD
OIL
SP500_Return
NASDAQ_Return
GOLD_Return
OIL_Return
SP500_Log_Return
NASDAQ_Log_Return
GOLD_Log_Return
OIL_Log_Return
SP500_Volatility_10
SP500_Volatility_20
NASDAQ_Volatility_20
SP500_Momentum_5
SP500_Momentum_10
SP500_Momentum_20
SP500_MA20
SP500_MA50
SP500_MA20_Distance
SP500_MA50_Distance
VIX_Change
VIX_MA10
VIX_Ratio

Sample:
                  SP500        VIX       NASDAQ        GOLD        OIL  \
Date                                                                     
2000-11-08  1409.280029  25.660000  3231.699951  265.000000  33.290001   
2000-11-09  1400.140015  27.200001  3200.350098  265.799988  33.950001   
2000-11

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



Saved: data/market_data_processed.csv
